# مشروع Database Replication على Google Colab
PostgreSQL Streaming Replication: سيرفر Primary (المنفذ 5432) + سيرفر Replica (المنفذ 5433) داخل نفس جلسة Colab.

شغّلي الخلايا بالترتيب (Shift+Enter).

## 1) تثبيت PostgreSQL داخل Colab

In [1]:
%%bash
apt-get -qq update
apt-get -qq install -y postgresql postgresql-contrib > /dev/null
echo "تم التثبيت"

تم التثبيت


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import os, subprocess
VER = sorted(os.listdir('/usr/lib/postgresql'))[-1]
os.environ['VER'] = VER
print('PostgreSQL version:', VER)

PostgreSQL version: 16


## 2) تشغيل الـ Primary وإنشاء مستخدم النسخ

In [3]:
%%bash
pg_ctlcluster $VER main start || true
sleep 3
su postgres -c "psql -c \"CREATE ROLE replicator WITH REPLICATION LOGIN PASSWORD 'rep123';\"" || true
su postgres -c "psql -c 'SHOW wal_level;'"

CREATE ROLE
 wal_level 
-----------
 replica
(1 row)



## 3) بناء الـ Replica من نسخة أساسية (pg_basebackup) وتشغيله

In [4]:
%%bash
pg_ctlcluster $VER replica stop 2>/dev/null || true
pg_dropcluster $VER replica 2>/dev/null || true
pg_createcluster $VER replica -p 5433
rm -rf /var/lib/postgresql/$VER/replica
su postgres -c "PGPASSWORD=rep123 pg_basebackup -h 127.0.0.1 -p 5432 -U replicator -D /var/lib/postgresql/$VER/replica -R -X stream -P"
pg_ctlcluster $VER replica start
sleep 3
pg_lsclusters

Creating new PostgreSQL cluster 16/replica ...
/usr/lib/postgresql/16/bin/initdb -D /var/lib/postgresql/16/replica --auth-local peer --auth-host scram-sha-256 --no-instructions
The files belonging to this database system will be owned by user "postgres".
This user must also own the server process.

The database cluster will be initialized with locale "en_US.UTF-8".
The default database encoding has accordingly been set to "UTF8".
The default text search configuration will be set to "english".

Data page checksums are disabled.

fixing permissions on existing directory /var/lib/postgresql/16/replica ... ok
creating subdirectories ... ok
selecting dynamic shared memory implementation ... posix
selecting default max_connections ... 100
selecting default shared_buffers ... 128MB
selecting default time zone ... Etc/UTC
creating configuration files ... ok
running bootstrap script ... ok
performing post-bootstrap initialization ... ok
syncing data to disk ... ok
Ver Cluster Port Status Owner 

waiting for checkpoint
23094/23094 kB (100%), 0/1 tablespace
23094/23094 kB (100%), 1/1 tablespace


## 4) التحقق من حالة النسخ

In [5]:
%%bash
echo "--- على الـ Primary: الـ Replicas المتصلة ---"
su postgres -c "psql -p 5432 -c 'SELECT client_addr, state, sync_state FROM pg_stat_replication;'"
echo "--- على الـ Replica: هل هو بوضع استعادة؟ (t = نعم) ---"
su postgres -c "psql -p 5433 -c 'SELECT pg_is_in_recovery();'"

--- على الـ Primary: الـ Replicas المتصلة ---
 client_addr |   state   | sync_state 
-------------+-----------+------------
 127.0.0.1   | streaming | async
(1 row)

--- على الـ Replica: هل هو بوضع استعادة؟ (t = نعم) ---
 pg_is_in_recovery 
-------------------
 t
(1 row)



## 5) اختبار: إدخال بيانات على Primary وقراءتها من Replica

In [6]:
%%bash
su postgres -c "psql -p 5432 -c 'CREATE TABLE IF NOT EXISTS students(id SERIAL PRIMARY KEY, name TEXT);'"
su postgres -c "psql -p 5432 -c \"INSERT INTO students(name) VALUES ('Ali'),('Sara'),('Omar');\""
sleep 2
echo "--- القراءة من الـ Replica ---"
su postgres -c "psql -p 5433 -c 'SELECT * FROM students;'"

CREATE TABLE
INSERT 0 3
--- القراءة من الـ Replica ---
 id | name 
----+------
  1 | Ali
  2 | Sara
  3 | Omar
(3 rows)



## 6) اختبار: الكتابة على الـ Replica مرفوضة (Read-Only)

In [7]:
%%bash
su postgres -c "psql -p 5433 -c \"INSERT INTO students(name) VALUES ('Test');\"" || echo "متوقع: الـ Replica للقراءة فقط"

متوقع: الـ Replica للقراءة فقط


ERROR:  cannot execute INSERT in a read-only transaction


## 7) محاكاة الفشل (Failover): إيقاف الـ Primary وترقية الـ Replica

In [8]:
%%bash
pg_ctlcluster $VER main stop
pg_ctlcluster $VER replica promote
sleep 3
su postgres -c "psql -p 5433 -c 'SELECT pg_is_in_recovery();'"
su postgres -c "psql -p 5433 -c \"INSERT INTO students(name) VALUES ('NewPrimaryTest');\""
su postgres -c "psql -p 5433 -c 'SELECT * FROM students;'"

 pg_is_in_recovery 
-------------------
 f
(1 row)

INSERT 0 1
 id |      name      
----+----------------
  1 | Ali
  2 | Sara
  3 | Omar
 34 | NewPrimaryTest
(4 rows)

